# ATML PA2: Task 1 - Direct Preference Optimization (DPO)

This notebook runs all Task 1 experiments on a Kaggle GPU environment (T4 x 1 or P100):
1. **Environment & Assets Setup**
2. **Step 1: Standard DPO (1 epoch)**
3. **Step 1: Standard DPO Evaluation (Held-out pairs, KL, Reward Model score, Length)**
4. **Step 2: Regularization-Strength Ablation ($\beta \in \{0.03, 0.10, 0.30\}$)**
5. **Step 3: Length-Confounding Study (Stratified evaluation & Word-limit compliance)**
6. **Results Packaging & Inspection**

## 1. Verify GPU and Setup Environment

In [ ]:
!nvidia-smi

## 2. Clone Your GitHub Repository
Replace `YOUR_GITHUB_USERNAME` and repo name below with your repository.

In [ ]:
# Set your repository URL here:
# e.g., https://github.com/YOUR_USERNAME/ATML-PA2-LLM-PostTraining.git
# Or if using a Personal Access Token (PAT) for private repo:
# REPO_URL = "https://<YOUR_GITHUB_TOKEN>@github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"
REPO_URL = "https://github.com/YOUR_USERNAME/ATML-PA2-LLM-PostTraining.git"

import os
repo_name = REPO_URL.split('/')[-1].replace('.git', '')
if not os.path.exists(repo_name):
    !git clone $REPO_URL
%cd $repo_name
!git status

## 3. Install Requirements & Download Assets

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment

In [ ]:
# Download course assets from Hugging Face and validate
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Run Task 1 - Step 1: Standard DPO Training (1 Epoch)
Trains on `dpo_standard_train.jsonl` with $\beta=0.10$ and saves adapter to `outputs/task1_dpo/standard`.

In [ ]:
!python -m task1_dpo.train --config configs/dpo.yaml --run-name standard

## 5. Run Task 1 - Step 1: Standard DPO Evaluation
Evaluates the standard DPO adapter on held-out pairs (`dpo_standard_eval.jsonl`), generates responses, scores with Reward Model, and logs KL drift and response length statistics.

In [ ]:
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard

## 6. Run Task 1 - Step 2: Regularization-Strength Ablation ($\beta \in \{0.03, 0.10, 0.30\}$)
Trains short forks (600 examples) for each $\beta$ and evaluates all of them under identical protocols.

In [ ]:
!python -m task1_dpo.ablate_beta --config configs/dpo.yaml

## 7. Run Task 1 - Step 3: Length-Confounding Study
Trains length-balanced DPO model on `dpo_length_balanced_train.jsonl`, then compares standard vs. length-balanced DPO on:
1. Length-stratified held-out pairs (preferred-longer, matched-length, rejected-longer)
2. Explicit word-limit compliance (`data/word_limit_prompts.jsonl`)

In [ ]:
!python -m task1_dpo.analyze_length --config configs/dpo.yaml

## 8. View and Inspect All Generated Results

In [ ]:
import pandas as pd
import json
from pathlib import Path

res_dir = Path("results/task1_dpo")

print("=== BETA ABLATION SUMMARY ===")
beta_csv = res_dir / "beta_ablation_summary.csv"
if beta_csv.exists():
    display(pd.read_csv(beta_csv))

print("\n=== STRATIFIED LENGTH EVALUATION ===")
len_csv = res_dir / "length_confounding_analysis.csv"
if len_csv.exists():
    display(pd.read_csv(len_csv))

print("\n=== WORD LIMIT COMPLIANCE ===")
word_csv = res_dir / "word_limit_compliance.csv"
if word_csv.exists():
    display(pd.read_csv(word_csv))

## 9. Package Results for Download or GitHub Push
Creates a zip file containing all JSON/CSV results and adapters so you can download them directly from Kaggle or push them back to your repo.

In [ ]:
# Zip results directory (small JSON and CSV files)
!zip -r task1_results.zip results/task1_dpo
print("task1_results.zip ready in Kaggle working directory!")

# Optional: Push results back to your GitHub repository
# !git config --global user.email "your_email@example.com"
# !git config --global user.name "Your Name"
# !git add results/task1_dpo
# !git commit -m "Add Task 1 DPO results and logs"
# !git push